# Seed Azure SQL Database

Loads this sector's CSVs from the lakehouse into the provisioned **Azure SQL
Database** so it can play the role of an operational source system.

The table list, primary keys, and column types come from the sector's
`mirroring.json` spec (injected at deploy time). Tables are created with
**primary keys**, a prerequisite for Fabric Database Mirroring.

Authentication is **Microsoft Entra-only** (no SQL password): this notebook runs
as you, the server's Entra admin, and also grants the Fabric **workspace identity**
the permissions Mirroring needs to read the source database.

In [ ]:
import base64, json, time
import notebookutils

# The per-sector spec is injected by the deployer as base64 of demos/<demo>/mirroring.json
SPEC = json.loads(base64.b64decode("{{MIRRORING_SPEC_B64}}").decode("utf-8"))

def _sql_access_token():
    errors = []
    for audience in ("https://database.windows.net/", "https://database.windows.net"):
        try:
            tok = notebookutils.credentials.getToken(audience)
            if tok:
                return tok
        except Exception as e:  # noqa: BLE001
            errors.append(f"{audience}: {e}")
    raise RuntimeError("Could not acquire an Azure SQL access token via "
                       "notebookutils.credentials.getToken: " + "; ".join(errors))

ROW_CAP = int("{{ROW_CAP}}")
SQL_SERVER = "{{SQL_SERVER}}"
SQL_DATABASE = "{{SQL_DATABASE}}"
WORKSPACE_IDENTITY_NAME = "{{WORKSPACE_IDENTITY_NAME}}"
WORKSPACE_IDENTITY_APP_ID = "{{WORKSPACE_IDENTITY_APP_ID}}"
DATA_SOURCE_PATH = "{{DATA_SOURCE_PATH}}"

ACCESS_TOKEN = _sql_access_token()

JDBC_URL = (
    f"jdbc:sqlserver://{SQL_SERVER}:1433;database={SQL_DATABASE};"
    "encrypt=true;trustServerCertificate=false;loginTimeout=60;"
)
JDBC_PROPS = {
    "accessToken": ACCESS_TOKEN,
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver",
}

def sql_connection(database=None):
    """Open a raw JDBC connection using the Entra access token."""
    url = (f"jdbc:sqlserver://{SQL_SERVER}:1433;database={database or SQL_DATABASE};"
           "encrypt=true;trustServerCertificate=false;loginTimeout=60;")
    jvm = spark._sc._jvm
    props = jvm.java.util.Properties()
    props.setProperty("accessToken", ACCESS_TOKEN)
    jvm.java.lang.Class.forName("com.microsoft.sqlserver.jdbc.SQLServerDriver")
    return jvm.java.sql.DriverManager.getConnection(url, props)

print(f"Target: {SQL_SERVER}/{SQL_DATABASE} (Microsoft Entra auth)")
print(f"Tables to mirror: {[t['name'] for t in SPEC['tables']]}")


In [ ]:
# Build each SQL table from the spec and load its CSV (deterministic types).
from pyspark.sql.functions import col, to_date, to_timestamp

def spark_type_to_sql(dt):
    """Safe SQL Server type for a column with no explicit spec type."""
    t = dt.simpleString()
    if t in ("tinyint", "smallint", "int"):
        return "INT"
    if t in ("bigint", "long"):
        return "BIGINT"
    if t in ("float", "double"):
        return "FLOAT"
    if t.startswith("decimal"):
        return t.upper()
    if t == "boolean":
        return "BIT"
    if t == "date":
        return "DATE"
    if t == "timestamp":
        return "DATETIME2(3)"
    return None  # string / other: sized from the data below

def cast_column(df, name, sqltype):
    """Cast a column to match an explicit spec type so the load is deterministic."""
    up = sqltype.upper()
    if up.startswith("DATETIME") :
        return df.withColumn(name, to_timestamp(col(name)))
    if up == "DATE":
        return df.withColumn(name, to_date(col(name)))
    if up.startswith("DECIMAL"):
        return df.withColumn(name, col(name).cast(up.lower()))
    if up in ("INT", "INTEGER"):
        return df.withColumn(name, col(name).cast("int"))
    if up == "BIGINT":
        return df.withColumn(name, col(name).cast("long"))
    if up == "FLOAT":
        return df.withColumn(name, col(name).cast("double"))
    if up == "BIT":
        return df.withColumn(name, col(name).cast("boolean"))
    return df  # strings: leave as-is

def load_table(tbl):
    name = tbl["name"]
    pk = tbl["primaryKey"]
    overrides = tbl.get("columnTypes", {})
    df = (spark.read.format("csv").option("header", True).option("inferSchema", True)
          .load(f"{DATA_SOURCE_PATH}/{tbl['sourceCsv']}"))
    if ROW_CAP and df.count() > ROW_CAP:
        df = df.limit(ROW_CAP)
    for cname, ctype in overrides.items():
        if cname in df.columns:
            df = cast_column(df, cname, ctype)
    df = df.dropDuplicates(pk)
    df.cache()
    schema = {f.name: f.dataType for f in df.schema.fields}
    col_defs = []
    for cname in df.columns:
        if cname in overrides:
            sqltype = overrides[cname]
        else:
            inferred = spark_type_to_sql(schema[cname])
            if inferred is None:
                maxlen = df.selectExpr(f"max(length(cast(`{cname}` as string)))").collect()[0][0] or 1
                size = min(max(int(maxlen) * 2, 50), 4000)
                sqltype = f"NVARCHAR({size})"
            else:
                sqltype = inferred
        null_clause = "NOT NULL" if cname in pk else "NULL"
        col_defs.append(f"    [{cname}] {sqltype} {null_clause}")
    pk_cols = ", ".join(f"[{c}]" for c in pk)
    ddl = (
        f"IF OBJECT_ID('dbo.{name}','U') IS NOT NULL DROP TABLE dbo.{name};\n"
        f"CREATE TABLE dbo.{name} (\n" + ",\n".join(col_defs) +
        f",\n    CONSTRAINT pk_{name} PRIMARY KEY ({pk_cols})\n);"
    )
    conn = sql_connection()
    stmt = conn.createStatement()
    try:
        for batch in [b.strip() for b in ddl.split(";") if b.strip()]:
            stmt.execute(batch)
    finally:
        stmt.close(); conn.close()
    df.write.mode("append").jdbc(JDBC_URL, f"dbo.{name}", properties=JDBC_PROPS)
    n = df.count()
    df.unpersist()
    print(f"dbo.{name}: created with PK {pk}, loaded {n:,} rows")
    return n

for tbl in SPEC["tables"]:
    load_table(tbl)
print("All tables created with primary keys and loaded")

In [ ]:
# Grant the Fabric workspace identity the permissions Mirroring requires.
# Runs as you (the Entra admin). The workspace identity is a contained Entra
# user in the database; no SQL login or password is involved.
import uuid

wi = WORKSPACE_IDENTITY_NAME.replace("]", "]]")  # escape ] for bracket-quoting
app_id = WORKSPACE_IDENTITY_APP_ID.strip()

def _exec(conn, sql):
    stmt = conn.createStatement()
    try:
        stmt.execute(sql)
    finally:
        stmt.close()

def _is_principal_not_found(err: Exception) -> bool:
    m = str(err).lower()
    # SQL 33134 / "Principal '<name>' could not be ... found / resolved" → either
    # the identity hasn't propagated in Entra yet, or the server can't read the
    # directory. Either way the retry below gives propagation a chance.
    return (
        "principal" in m and ("not be" in m or "could not" in m or "not found" in m or "does not exist" in m)
    ) or "33134" in m

# Preferred path: create the contained user BY SID. The Fabric workspace identity
# is an Entra service principal whose SID is its Application (client) ID in
# little-endian byte order. Creating the user this way maps the principal WITHOUT
# Azure SQL having to call Microsoft Graph — which is what `FROM EXTERNAL PROVIDER`
# does and which fails in governed tenants where the SQL server's identity lacks
# the Entra "Directory Readers" role. It's also instant (no propagation wait).
if app_id:
    sid = "0x" + uuid.UUID(app_id).bytes_le.hex().upper()
    create_user_sql = (
        f"IF NOT EXISTS (SELECT 1 FROM sys.database_principals WHERE name = N'{WORKSPACE_IDENTITY_NAME}') "
        f"CREATE USER [{wi}] WITH SID = {sid}, TYPE = E;"
    )
    conn = sql_connection()
    try:
        _exec(conn, create_user_sql)
        print(f"Workspace identity user created by SID ({sid})")
    finally:
        conn.close()
else:
    # Fallback: no Application ID was injected, so resolve the principal via Entra
    # with `FROM EXTERNAL PROVIDER`. A brand-new principal can take a minute or two
    # to replicate, so poll on a short, backing-off cadence (~5 min budget) and
    # only retry on principal-not-found; surface anything else immediately.
    create_user_sql = (
        f"IF NOT EXISTS (SELECT 1 FROM sys.database_principals WHERE name = N'{WORKSPACE_IDENTITY_NAME}') "
        f"CREATE USER [{wi}] FROM EXTERNAL PROVIDER;"
    )
    _intervals = [5, 5, 5, 8, 8, 10, 10, 12, 15, 15, 20, 20, 25, 30, 30, 30, 30, 30]
    _t0 = time.time()
    last_err = None
    for attempt, _wait in enumerate(_intervals, start=1):
        conn = sql_connection()
        try:
            _exec(conn, create_user_sql)
            last_err = None
            print(f"Workspace identity resolved after {int(time.time() - _t0)}s "
                  f"(attempt {attempt}/{len(_intervals)})")
            break
        except Exception as e:  # noqa: BLE001
            last_err = e
            if not _is_principal_not_found(e):
                raise
            print(f"[{attempt}/{len(_intervals)}] identity not resolvable yet "
                  f"({int(time.time() - _t0)}s elapsed) — retrying in {_wait}s…")
            time.sleep(_wait)
        finally:
            conn.close()
    if last_err is not None:
        raise RuntimeError(
            "The Fabric workspace identity never became resolvable in Azure SQL within "
            f"~{int(time.time() - _t0)}s. This is usually slow Microsoft Entra propagation — "
            "retry the deployment. Last error: " + str(last_err)[:200]
        )

conn = sql_connection()
try:
    for perm in ("SELECT", "ALTER ANY EXTERNAL MIRROR",
                 "VIEW DATABASE PERFORMANCE STATE", "VIEW DATABASE SECURITY STATE"):
        _exec(conn, f"GRANT {perm} TO [{wi}];")
finally:
    conn.close()

print(f"Workspace identity '{WORKSPACE_IDENTITY_NAME}' granted mirroring permissions")


In [ ]:
# Verify row counts straight from SQL
for tbl in SPEC["tables"]:
    name = tbl["name"]
    n = (spark.read.jdbc(JDBC_URL, f"(SELECT COUNT(*) AS n FROM dbo.{name}) q",
                         properties=JDBC_PROPS).collect()[0]["n"])
    print(f"dbo.{name}: {n:,} rows")
print("\nAzure SQL database seeded. Ready for mirroring.")